In [1]:
%%capture
!pip install unsloth

!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

This code import a pre-trained LLM and fine-tune it using a given dataset.
Some part of this code are taken from the github repository of unsloth
LINK: https://github.com/unslothai/unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 
dtype = None # None for auto detection. Float16 for Tesla T4
load_in_4bit = False # Use 4bit quantization to reduce memory usage.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct", # put model name or path 
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.6.12: Fast Llama patching. Transformers: 4.53.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.3.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [ ]:
model = FastLanguageModel.get_peft_model(
    model, # Base model to which PEFT (LoRA) will be applied
    r = 16, # # LoRA rank: determines the dimension of the low-rank matrices. NOTA: Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Whether to train biases; "none" disables it for efficiency
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407, # Seed for reproducibility
    use_rslora = False,    # If True, enables rank-stabilized LoRA (an advanced LoRA variant)
    loftq_config = None, # Optional config for LoftQ, a quantization-aware training method
)

Unsloth 2025.6.12 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [ ]:
from unsloth.chat_templates import get_chat_template   # Import function to apply chat formatting to tokenizer

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)
FastLanguageModel.for_inference(model) 

messages = [
    {"role": "prompt", "content": "How are you??"},
]
inputs = tokenizer.apply_chat_template(
    messages, # The message list to format and tokenize
    tokenize = True, # Tokenize the formatted message
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt", #PyTorch tensors
).to("cuda")

# Generate the attention mask
attention_mask = inputs.ne(tokenizer.pad_token_id).long().to("cuda")


from transformers import TextStreamer  # Import streamer to display generated text in real time
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, attention_mask = attention_mask, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

I'm just a language model, I don't have emotions or feelings, but thank you for asking. I'm here to help with any questions or tasks you may have, so feel free to ask me anything. How can I assist you today?<|eot_id|>


In [7]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)

In [8]:
import pandas as pd
df = pd.read_csv("/content/LLM_train.csv")

In [9]:
df.head()

,prompt,answer
0,You mustve had your hands full.,That I did. That I did.
1,So lets talk a little bit about your duties.,My duties? All right.
2,"Now youll be heading a whole division, so you...",I see.
3,But therell be perhaps 30 people under you so...,Good to know.
4,We can go into detail,No dont I beg of you!


In [10]:
df = df.dropna()

In [ ]:
from datasets import Dataset
df["conversations"] = df.apply(
    lambda x: [
        {"content": x["prompt"], "role": "user"},
        {"content": x["answer"], "role": "assistant"}
    ], axis=1
)

dataset = Dataset.from_pandas(df.drop(columns=["prompt", "answer"]))

In [12]:
dataset

Dataset({
    features: ['conversations'],
    num_rows: 930
})

In [13]:
dataset['conversations'][0]

[{'content': 'You must\x92ve had your hands full.', 'role': 'user'},
 {'content': 'That I did. That I did.', 'role': 'assistant'}]

In [ ]:
def formatting_prompts_func(examples):      # Function to format the dataset for training
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }
pass

from unsloth.chat_templates import standardize_sharegpt
dataset = standardize_sharegpt(dataset)
dataset = dataset.map(formatting_prompts_func, batched = True,)

Unsloth: Standardizing formats (num_proc=2):   0%|          | 0/930 [00:00<?, ? examples/s]

Map:   0%|          | 0/930 [00:00<?, ? examples/s]

In [15]:
dataset[5]["conversations"]

[{'content': 'All right then, we\x92ll have a definite answer for you on Monday, but I think I can say with some confidence, you\x92ll fit in well here.',
  'role': 'user'},
 {'content': 'Really?!', 'role': 'assistant'}]

In [16]:
dataset[5]["text"]

'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 July 2024\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nAll right then, we\x92ll have a definite answer for you on Monday, but I think I can say with some confidence, you\x92ll fit in well here.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nReally?!<|eot_id|>'

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    dataset_num_proc = 2,
    packing = False, # useful feature for short sequences
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5, # Linear warmup: start with small LR for first 5 steps
        max_steps = 20,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01, # Apply 0.01 L2 regularization to weights
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", 
    )
)

Unsloth: Tokenizing ["text"]:   0%|          | 0/930 [00:00<?, ? examples/s]

In [21]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

Map (num_proc=2):   0%|          | 0/930 [00:00<?, ? examples/s]

In [22]:
#@title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.741 GB.
6.779 GB of memory reserved.


In [28]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 930 | Num Epochs = 1 | Total steps = 20
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Step,Training Loss
1,2.249800
2,2.779300
3,2.134500
4,2.549900
5,2.211500
6,2.362100
7,2.624700
8,2.585700
9,2.034200
10,2.067200


In [29]:
#60 epochs

FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [
    {"role": "user", "content": "How are you?"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

# Generate the attention mask
attention_mask = inputs.ne(tokenizer.pad_token_id).long().to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, attention_mask = attention_mask, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

Im great! Thanks for asking.<|eot_id|>


In [ ]:

# TEST SET VALIDATION

import os
import pandas as pd
from tqdm import tqdm

test_file = "/content/LLM_test.csv"  # path test set
output_file = "/content/LLM_test_with_generated.csv"  # file to save results

if os.path.exists(test_file):
    #print("\nLoading test dataset")
    test_df = pd.read_csv(test_file).dropna(subset=["prompt", "answer"])  #drop rows with NaN in 'prompt' or 'answer'

    generated_responses = []

    #print("Generating responses")
    for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
        # Loop through each row in the test DataFrame, showing progress with tqdm
        prompt = row["prompt"]
        actual_answer = row["answer"]

        messages = [
            {"content": prompt, "role": "user"}
        ]

        # Prepare the input for the tokenizer and model
        inputs = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt"
        ).to("cuda")

        attention_mask = inputs.ne(tokenizer.pad_token_id).long().to("cuda")

        # Generate the response
        outputs = model.generate(
            input_ids=inputs,
            attention_mask=attention_mask,
            max_new_tokens=128,
            temperature=0.7,
            do_sample=True
        )
        # Decode the generated response
        # Skip special tokens like <|start_header_id|>assistant<|end_header_id|>
        # and extract the actual response text
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        generated_text = response.split("assistant\n\n")[-1].strip()

        generated_responses.append(generated_text)

    # Adding the column with the generated response to the dataframe
    test_df["generated"] = generated_responses

    test_df.to_csv(output_file, index=False)

    print(f"\nSaved generated responses to {output_file}")
else:
    print(f"Test file {test_file} not found. Skipping generation.")



Loading test dataset...
Generating responses for all prompts...


100%|██████████| 266/266 [02:03<00:00,  2.16it/s]


Saved generated responses to /content/LLM_test_with_generated.csv


In [ ]:
model.save_pretrained("/content/trained_model", tokenizer)

In [ ]:
#cella per riprendere i pesi


from unsloth import FastLanguageModel
import torch

# Define the path to the saved model
model_path = "/content/trained_model" 

# Choose your desired settings (must match the original training setup)
max_seq_length = 2048
dtype = None # specify if known, e.g., "float16" or "bfloat16"
load_in_4bit = False 

# Loading the trained model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_path, # Use the path 
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

FastLanguageModel.for_inference(model)

# Send a prompt to the model
messages = [
    {"role": "prompt", "content": "Tell me something interesting."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

# Generate the attention mask
attention_mask = inputs.ne(tokenizer.pad_token_id).long().to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, attention_mask = attention_mask, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True, temperature = 0.7, min_p = 0.5)


In [ ]:
# Save the model in GGUF format for Ollama
model.save_pretrained_gguf("model", tokenizer, quantization_method = "f16")
'''
!ollama create my-unsloth-llama3 -f ./Modelfile
!ollama serve # In a separate terminal or process
!ollama run my-unsloth-llama3 "Your prompt here"
'''